In [1]:
from Environment import UAVEnv
import gymnasium as gym
import torch as th
from torch import nn

In [2]:
env= UAVEnv()
env.reset()
def make_env():
    return UAVEnv()

In [3]:
env.step(1)

({'needs': array([1.        , 0.81994551, 1.        , 1.        , 1.        ,
         1.        , 1.        , 1.        ]),
  'directions': array([-0.25701804, -0.30846223, -0.34285197, -0.07911743, -0.07898656,
          0.17396206,  0.36522352, -0.48368678]),
  'distance': array([0.63787914, 0.38816611, 0.72785026, 0.44789968, 0.26416378,
         0.31050705, 0.35737109, 0.2242696 ]),
  'user_satisfied': array([False, False, False, False, False, False, False, False])},
 -0.7374430888762428,
 False,
 False,
 {'JFI': 0.8749999843750004,
  'Bandwidth': 1.1674595320384342,
  'Thr_fairness': -66.18687837671776})

In [4]:
env.get_action_mask()

[array([ True,  True,  True,  True,  True,  True,  True,  True])]

In [5]:
from sb3_contrib.common.maskable.callbacks import MaskableEvalCallback
from stable_baselines3.common.logger import Logger, KVWriter
import numpy as np

class TensorboardEvalCallback(MaskableEvalCallback):
    def __init__(self, eval_env, log_path, eval_freq=10000, n_eval_episodes=5, deterministic=True):
        super().__init__(
            eval_env=eval_env,
            best_model_save_path=log_path,
            log_path=log_path,
            eval_freq=eval_freq,
            n_eval_episodes=n_eval_episodes,
            deterministic=deterministic
        )

    def _on_step(self) -> bool:
        result = super()._on_step()

        if self.eval_freq > 0 and self.n_calls % self.eval_freq == 0:
            # Also log results to TensorBoard
            self.logger.record("eval/mean_reward", self.last_mean_reward)
            self.logger.record("eval/std_reward", np.std(self.evaluations_results[-1]))
        return result

In [6]:
import torch as th

def make_edge_index(num_users):
    # Fully connected, directed edges (i→j for all i≠j)
    senders, receivers = zip(
        *[(i, j) for i in range(num_users) for j in range(num_users) if i != j]
    )
    edge_index = th.tensor([senders, receivers], dtype=th.long)  # (2, E)
    return edge_index


In [7]:
import torch.nn.functional as F
from torch import nn

class GraphConv(nn.Module):
    def __init__(self, in_feats, out_feats):
        super().__init__()
        self.lin = nn.Linear(in_feats, out_feats, bias=False)
        self.root_lin = nn.Linear(in_feats, out_feats, bias=True)

    def forward(self, x, edge_index):
        # x: (batch, num_nodes, in_feats)
        # edge_index: (2, E)
        batch_size, num_nodes, _ = x.size()
        # 1) Linear transform of neighbors
        h = self.lin(x)  # (batch, num_nodes, out_feats)
        # 2) For message passing, we need to gather neighbor messages:
        src, dst = edge_index
        # expand for batch:
        h_src = h[:, src, :]        # (batch, E, out_feats)
        # aggregate by summing into each destination node:
        agg = th.zeros(batch_size, num_nodes, h.size(-1), device=x.device)
        agg = agg.index_add(1, dst, h_src)  # sum over edges
        # 3) Combine with transformed self-features (root)
        out = agg + self.root_lin(x)
        return F.relu(out)


In [8]:
import torch as th
import torch.nn as nn
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor

class GNNFeatureExtractor(BaseFeaturesExtractor):
    def __init__(self, observation_space, embed_dim=128, num_layers=2, k_knn=None):
        # We’ll output a vector of size embed_dim
        super().__init__(observation_space, features_dim=embed_dim)
        self.num_users = observation_space['needs'].shape[0]
        self.in_dim = 4  # needs, delay, direction, distance, satisfied_flag
        
        # Build (static) edge_index—for fully connected graph
        self.register_buffer('edge_index', make_edge_index(self.num_users))
        # If you prefer kNN, compute it here once and register as buffer
        
        # Input projection
        self.input_lin = nn.Linear(self.in_dim, embed_dim)
        
        # Stacked GraphConv layers
        self.convs = nn.ModuleList([
            GraphConv(embed_dim, embed_dim) for _ in range(num_layers)
        ])
        
        # Final readout (pooling + projection)
        self.readout = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.ReLU(),
            nn.Linear(embed_dim, embed_dim),
        )

    def forward(self, obs):
        # Build node features: (batch, num_users, 5)
        # Create satisfied_flag = 1 if needs <= 0
        sat = (obs['needs'] <= 0).float()
        # Stack along a _new_ last dimension:
        user_feats = th.stack([
            obs['needs'],
            #obs['delay'],
            obs['directions'],
            obs['distance'],
            sat
        ], dim=-1)   # -> (batch, num_users, 5)
        
        # Input projection
        h = self.input_lin(user_feats)       # (batch, num_users, embed_dim)
        # GNN layers
        for conv in self.convs:
            h = conv(h, self.edge_index)
        # Readout: mean-pool over nodes, then MLP
        h = h.mean(dim=1)           # (batch, embed_dim)
        return self.readout(h)      # (batch, embed_dim)


In [9]:
## defining test environment and test model
from stable_baselines3.common.vec_env import DummyVecEnv

# Create one test environment (no need for parallel here)
test_env = DummyVecEnv([make_env])


In [10]:
import os
import json
from datetime import datetime

# Create a unique experiment directory
run_name = f"ppo_run_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
run_dir = os.path.join("ppo_experiments_GNN", run_name)
log_dir = os.path.join(run_dir, "tensorboard")
os.makedirs(log_dir, exist_ok=True)

In [11]:
# Flatten or stringify any non-JSON-serializable items
num_envs=1
hyperparams = {
    "policy_kwargs": {
        "net_arch": {
            "pi": [512, 256, 128],
            "vf": [512, 128, 64]
        },
        "optimizer_kwargs": {
            "weight_decay": 1e-4
        }
    },
    "n_steps": 256,
    "batch_size": 64,
    # "learning_rate": {
    #     "type": "exponential_schedule",
    #     "initial_value": 3e-4,
    #     "decay_rate": 0.9
    # },
    "learning_rate": 1e-4,
    "clip_range": 0.1,
    "ent_coef": 0.01,
    "gae_lambda":0.92,
    "vf_coef": 0.2,
    "n_epochs": 10,
    "gamma":0.9995,
    "num_envs": num_envs,
    "policy_class": "CustomPolicy"  # Store as string instead of object
}


with open(os.path.join(run_dir, "hyperparameters.json"), "w") as f:
    json.dump(hyperparams, f, indent=4)

In [12]:
from sb3_contrib.common.wrappers import ActionMasker
#from stable_baselines3.common.vec_env import DummyVecEnv
def mask_fn(env):
    return [env.needs>env.progress]
env = ActionMasker(env, mask_fn)

## defining test environment and test model
test_env = ActionMasker(UAVEnv(), mask_fn)

In [13]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import gymnasium as gym
from gymnasium import spaces
import numpy as np

from sb3_contrib.ppo_mask import MaskablePPO
from sb3_contrib.common.maskable.utils import get_action_masks
from sb3_contrib.ppo_mask.policies import MultiInputPolicy
from stable_baselines3.common.vec_env import VecNormalize

policy_kwargs = dict(
    features_extractor_class=GNNFeatureExtractor,
    features_extractor_kwargs=dict(
        embed_dim=256,
        num_layers=3,
        # k_knn=3  # if you implement a kNN adjacency instead
    ),
      net_arch = dict(
    pi=[256,128],
    vf=[512,256,128] ,
    activation_fn=nn.ReLU
)
)

def exponential_schedule(initial_value, decay_rate=0.95):
    def lr_schedule(progress_remaining):
        return initial_value * (decay_rate ** (1 - progress_remaining))
    return lr_schedule

#vecenv = DummyVecEnv([make_env])
# vecenv = VecNormalize(
#     env,
#     norm_obs=True,
#     norm_reward=True,
#     clip_obs=10.0,
#     clip_reward=10.0,
#     # Only normalize these observation keys:
#     norm_obs_keys=["needs", "delay", "directions", "distance"],
# )

# Adjust PPO hyperparameters
model = MaskablePPO(
    "MultiInputPolicy",
    env,
    verbose=2,
    tensorboard_log=log_dir,
    policy_kwargs=policy_kwargs,
    n_steps=hyperparams["n_steps"],
    batch_size=hyperparams["batch_size"],
    learning_rate= exponential_schedule(hyperparams["learning_rate"], decay_rate=.8),
    clip_range=hyperparams["clip_range"],
    ent_coef=hyperparams["ent_coef"],
    vf_coef=hyperparams["vf_coef"],
    gae_lambda=hyperparams["gae_lambda"],
    n_epochs=hyperparams["n_epochs"],
    gamma=0.995  # Example value, you can change this
)


print(model.policy)

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
MaskableMultiInputActorCriticPolicy(
  (features_extractor): GNNFeatureExtractor(
    (input_lin): Linear(in_features=4, out_features=256, bias=True)
    (convs): ModuleList(
      (0-2): 3 x GraphConv(
        (lin): Linear(in_features=256, out_features=256, bias=False)
        (root_lin): Linear(in_features=256, out_features=256, bias=True)
      )
    )
    (readout): Sequential(
      (0): Linear(in_features=256, out_features=256, bias=True)
      (1): ReLU()
      (2): Linear(in_features=256, out_features=256, bias=True)
    )
  )
  (pi_features_extractor): GNNFeatureExtractor(
    (input_lin): Linear(in_features=4, out_features=256, bias=True)
    (convs): ModuleList(
      (0-2): 3 x GraphConv(
        (lin): Linear(in_features=256, out_features=256, bias=False)
        (root_lin): Linear(in_features=256, out_features=256, bias=True)
      )
    )
    (readout): Sequential(
      (0): L

In [ ]:
eval_callback = TensorboardEvalCallback(
    eval_env=test_env,
    log_path=run_dir,
    eval_freq=10_000,  # evaluate every N steps
    n_eval_episodes=5
)

model.learn(
    total_timesteps=500_000,
    tb_log_name="PPO_with_eval",
    callback=eval_callback
)
model_path = os.path.join(run_dir, "ppo_model_GNN")
model.save(model_path)

Logging to ppo_experiments_GNN/ppo_run_20250919_064925/tensorboard/PPO_with_eval_1
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 50.8     |
|    ep_rew_mean     | -20.9    |
| time/              |          |
|    fps             | 227      |
|    iterations      | 1        |
|    time_elapsed    | 1        |
|    total_timesteps | 256      |
---------------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 50.4         |
|    ep_rew_mean          | -20.9        |
| time/                   |              |
|    fps                  | 126          |
|    iterations           | 2            |
|    time_elapsed         | 4            |
|    total_timesteps      | 512          |
| train/                  |              |
|    approx_kl            | 0.0033775778 |
|    clip_fraction        | 0.113        |
|    clip_range           | 0.1          |
|    entropy_loss 

/usr/local/lib/python3.8/dist-packages/sb3_contrib/common/maskable/evaluation.py:72: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


Eval num_timesteps=10000, episode_reward=-22.91 +/- 1.18
Episode length: 50.80 +/- 1.83
-----------------------------------------
| eval/                   |             |
|    mean_ep_length       | 50.8        |
|    mean_reward          | -22.9       |
| time/                   |             |
|    total_timesteps      | 10000       |
| train/                  |             |
|    approx_kl            | 0.001783588 |
|    clip_fraction        | 0.0457      |
|    clip_range           | 0.1         |
|    entropy_loss         | -1.58       |
|    explained_variance   | 0.986       |
|    learning_rate        | 9.96e-05    |
|    loss                 | 0.0064      |
|    n_updates            | 390         |
|    policy_gradient_loss | -0.00734    |
|    value_loss           | 0.344       |
-----------------------------------------
New best mean reward!
---------------------------------
| eval/              |          |
|    mean_reward     | -22.9    |
|    std_reward      | 1.18     